### Trabalho Fase 1 do Curso de Pós-Graduação FIAP IA para Devs
#### Parte 1 - Dados Gerais - Redução do volume da base de dados

Fonte de dados escolhida: DATASUS  
Tipo de dados de origem: DBC

* Foram obtidos no site do SUS os dados de SIH/SUS - Sistema de Informações Hospitalares, arquivo RD - AIH Reduzida, do estado do Rio de Janeiro no ano de 2025.
* Os dados originais são disponibilizados em formato DBC e divididos por mês, com isso foi necessário:
    * Descompactá-los para o formato DBF
    * Realizar a conversão para o formato Parquet
        - Arquivos parquet conseguem otimizar o volume de dados facilitando seu tamanho e manipulação
    * Realizar o merge mês a mês para gerar o arquivo referente ao ano de 2025
    * O resultado foi o arquivo 2025_consolidado.parquet
    * Todas as ações anteriores não foram documentadas passo a passo aqui, pois não fazem parte do escopo geral da apresentação

---

## Sumário da Parte 1

**Objetivo geral - Redução do volume de dados**

Organizar a base SIH/SUS RD de 2025, avaliar sua estrutura inicial, manter apenas as colunas relevantes ao recorte analítico e gerar uma versão reduzida em Parquet para as próximas etapas de tratamento.

**Item 1 - Leitura estrutural da base consolidada**

- **1.1 - Carregamento e amostra inicial**
- **1.2 - Descrição estatística da base**
- **1.3 - Dimensões da base consolidada**
- **1.4 - Tipos de dados e uso de memória**

**Item 2 - Dicionário de Dados - SIH/SUS RD - AIH Reduzida**

- **2.1 - Consulta das colunas originais**
- **2.2 - Entendimento semântico dos campos**

**Item 3 - Remoção de colunas não utilizadas**

- **3.1 - Definição das colunas mantidas**
- **3.2 - Registro das colunas removidas**
- **3.3 - Geração da base reduzida**

**Item 4 - Validação da base reduzida**

- **4.1 - Conferência das colunas finais**
- **4.2 - Conferência do shape final**


#### Item 1 - Leitura estrutural da base consolidada

**Neste passo vamos:**

- **1.1 - Carregar e visualizar uma amostra da base consolidada** para confirmar que o arquivo `2025_consolidado.parquet` foi lido corretamente.
- **1.2 - Avaliar a descrição estatística geral** para observar distribuições, valores frequentes e comportamento inicial das colunas.
- **1.3 - Conferir as dimensões da base** para registrar quantidade de linhas e colunas antes da redução.
- **1.4 - Avaliar tipos de dados e uso de memória** para entender a estrutura inicial antes de selecionar colunas.


##### 1.1 - Carregamento e amostra inicial

Leitura do arquivo consolidado de 2025 e visualização das primeiras linhas para validar a estrutura carregada.


In [1]:
import pandas as pd

dados = pd.read_parquet('2025_consolidado.parquet')
dados.head()

,UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,...,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9,FONTE_ORC
0,330000,2025,01,07,,3324112102003,1,23066260,330455,20150219,...,0,0,0,0,0,0,0,0,0,NaN
1,330000,2025,01,07,,3324112102014,1,25070512,330170,20180804,...,0,0,0,0,0,0,0,0,0,NaN
2,330000,2025,01,03,42498717001046,3324108710329,1,25241390,330170,19670412,...,0,0,0,0,0,0,0,0,0,NaN
3,330000,2025,01,03,42498717001046,3324108710340,1,26195340,330045,19640923,...,2,0,0,0,0,0,0,0,0,NaN
4,330000,2025,01,03,42498717001046,3324108710351,1,27265550,330630,20040414,...,0,0,0,0,0,0,0,0,0,NaN


##### 1.2 - Descrição estatística da base

Avaliação geral das colunas numéricas e categóricas para entender valores, frequências e possíveis inconsistências iniciais.


In [2]:
dados.describe(include='all')

,UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,...,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9,FONTE_ORC
count,925240,925240,925240,925240,925240,925240,925240,925240,925240,925240,...,925240,925240,925240,925240,925240,925240,925240,925240,925240,777636
unique,83,1,12,13,176,918108,2,73321,827,36373,...,3,3,3,3,3,3,3,3,3,1
top,330455,2025,10,01,,3324106423297,1,28010000,330455,20250313,...,0,0,0,0,0,0,0,0,0,00
freq,284050,925240,80926,335867,264992,13,917577,12616,288172,184,...,722288,891839,909121,917761,922036,923862,924726,925100,925196,777636
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### 1.3 - Dimensões da base consolidada

Registro do volume original antes da remoção de colunas.


In [3]:
dados.shape

(925240, 114)

##### 1.4 - Tipos de dados e uso de memória

Verificação dos tipos inferidos para cada coluna, quantidade de valores não nulos e uso aproximado de memória.


In [4]:
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 925240 entries, 0 to 925239
Columns: 114 entries, UF_ZI to FONTE_ORC
dtypes: float64(19), int64(17), str(78)
memory usage: 1.0 GB


#### Item 2 - Dicionário de Dados - SIH/SUS RD - AIH Reduzida

**Neste passo vamos consultar o dicionário original para entender o significado das colunas antes da redução.**


| Ordem | Coluna | Descrição |
|---:|---|---|
| 1 | UF_ZI | Município gestor da internação, conforme codificação IBGE/DATASUS. |
| 2 | ANO_CMPT | Ano de competência/processamento da informação, no formato aaaa. |
| 3 | MES_CMPT | Mês de competência/processamento da informação, no formato mm. |
| 4 | ESPEC | Código da especialidade associada a AIH/internação. |
| 5 | CGC_HOSP | CNPJ/CGC do estabelecimento hospitalar. |
| 6 | N_AIH | Número da Autorização de Internacao Hospitalar (AIH). |
| 7 | IDENT | Identificacao do tipo de AIH. |
| 8 | CEP | CEP de residência do paciente. |
| 9 | MUNIC_RES | Código IBGE do município de residência do paciente. |
| 10 | NASC | Data de nascimento do paciente, no formato aaaammdd. |
| 11 | SEXO | Sexo do paciente: 1 masculino; 3 feminino. |
| 12 | UTI_MES_IN | Quantidade de dias de UTI no mês inicial da internação. |
| 13 | UTI_MES_AN | Quantidade de dias de UTI em mês anterior ao mes da alta. |
| 14 | UTI_MES_AL | Quantidade de dias de UTI no mês da alta. |
| 15 | UTI_MES_TO | Quantidade total de dias/atos de UTI. |
| 16 | MARCA_UTI | Marcador/tipo de UTI associado ao ato informado. |
| 17 | UTI_INT_IN | Dias em unidade intermediaria no mês inicial da internação. |
| 18 | UTI_INT_AN | Dias em unidade intermediaria em mês anterior ao mes da alta. |
| 19 | UTI_INT_AL | Dias em unidade intermediaria no mês da alta. |
| 20 | UTI_INT_TO | Total de dias em unidade intermediaria. |
| 21 | DIAR_ACOM | Quantidade de diarias de acompanhante. |
| 22 | QT_DIARIAS | Quantidade de diarias da internação. |
| 23 | PROC_SOLIC | Código do procedimento solicitado. |
| 24 | PROC_REA | Código do procedimento realizado. |
| 25 | VAL_SH | Valor dos serviços hospitalares. |
| 26 | VAL_SP | Valor dos serviços profissionais/prestados por terceiros. |
| 27 | VAL_SADT | Valor de serviços auxiliares de diagnóstico e terapia (SADT). |
| 28 | VAL_RN | Valor referente a recém-nato. |
| 29 | VAL_ACOMP | Valor referente a acompanhante. |
| 30 | VAL_ORTP | Valor de órtese e prótese. |
| 31 | VAL_SANGUE | Valor referente a sangue/hemoderivados. |
| 32 | VAL_SADTSR | Valor de SADT sem rateio. |
| 33 | VAL_TRANSP | Valor referente a transplante. |
| 34 | VAL_OBSANG | Valor referente a observação/anestesia ou sangue, conforme layout SIH. |
| 35 | VAL_PED1AC | Valor referente a pediatria/primeiro acompanhante, conforme layout SIH. |
| 36 | VAL_TOT | Valor total aprovado/pago da AIH. |
| 37 | VAL_UTI | Valor referente a UTI. |
| 38 | US_TOT | Valor total em dólar. |
| 39 | DT_INTER | Data da internação, no formato aaaammdd. |
| 40 | DT_SAIDA | Data da saída/alta, no formato aaaammdd. |
| 41 | DIAG_PRINC | Diagnostico principal informado, em código CID. |
| 42 | DIAG_SECUN | Diagnostico secundário informado, em código CID. |
| 43 | COBRANCA | Motivo de cobrança/saída/permanência da AIH. |
| 44 | NATUREZA | Natureza jurídica do hospital no layout legado. |
| 45 | NAT_JUR | Natureza jurídica do estabelecimento. |
| 46 | GESTAO | Tipo/código de gestão ou órgão emissor do gestor. |
| 47 | RUBRICA | Número da rubrica. |
| 48 | IND_VDRL | Indicador de exame VDRL. |
| 49 | MUNIC_MOV | Código IBGE do município do estabelecimento onde ocorreu a internação. |
| 50 | COD_IDADE | Unidade de medida usada para interpretar a idade. |
| 51 | IDADE | Idade do paciente conforme a unidade indicada em COD_IDADE. |
| 52 | DIAS_PERM | Quantidade de dias de permanência. |
| 53 | MORTE | Indicador de óbito. |
| 54 | NACIONAL | Código da nacionalidade do paciente. |
| 55 | NUM_PROC | Número do processamento. |
| 56 | CAR_INT | Carater da internação. |
| 57 | TOT_PT_SP | Total de pontos em serviços profissionais. |
| 58 | CPF_AUT | CPF do auditor/autorizador. |
| 59 | HOMONIMO | Indicador de homonímia do paciente. |
| 60 | NUM_FILHOS | Quantidade de filhos do paciente. |
| 61 | INSTRU | Grau de instrução do paciente. |
| 62 | CID_NOTIF | CID de notificação. |
| 63 | CONTRACEP1 | Metodo contraceptivo informado no campo 1. |
| 64 | CONTRACEP2 | Metodo contraceptivo informado no campo 2. |
| 65 | GESTRISCO | Indicador de gestante de alto risco. |
| 66 | INSC_PN | Número da gestante no pré-natal. |
| 67 | SEQ_AIH5 | Sequencial da AIH tipo 5, usada em longa permanência. |
| 68 | CBOR | Código da ocupação/CBO. |
| 69 | CNAER | Código de acidente de trabalho. |
| 70 | VINCPREV | Vínculo com a previdência. |
| 71 | GESTOR_COD | Motivo/código de autorização da AIH pelo gestor. |
| 72 | GESTOR_TP | Tipo de gestor. |
| 73 | GESTOR_CPF | CPF do gestor/autorizador. |
| 74 | GESTOR_DT | Data da autorização dada pelo gestor, no formato aaaammdd. |
| 75 | CNES | Código CNES do estabelecimento de saúde. |
| 76 | CNPJ_MANT | CNPJ da mantenedora do estabelecimento. |
| 77 | INFEHOSP | Indicador/status de infecção hospitalar. |
| 78 | CID_ASSO | CID associado/causa. |
| 79 | CID_MORTE | CID associado ao óbito. |
| 80 | COMPLEX | Complexidade do procedimento/internação. |
| 81 | FINANC | Tipo de financiamento. |
| 82 | FAEC_TP | Subtipo de financiamento FAEC. |
| 83 | REGCT | Regra contratual. |
| 84 | RACA_COR | Raca/cor do paciente. |
| 85 | ETNIA | Etnia do paciente, quando aplicavel. |
| 86 | SEQUENCIA | Sequencial da AIH na remessa. |
| 87 | REMESSA | Número/identificacao da remessa. |
| 88 | AUD_JUST | Justificativa do auditor para aceitacao da AIH sem Cartao Nacional de Saude. |
| 89 | SIS_JUST | Justificativa do estabelecimento/sistema para aceitacao da AIH sem Cartao Nacional de Saude. |
| 90 | VAL_SH_FED | Valor do complemento federal de serviços hospitalares, incluido no total da AIH. |
| 91 | VAL_SP_FED | Valor do complemento federal de serviços profissionais, incluido no total da AIH. |
| 92 | VAL_SH_GES | Valor do complemento do gestor para serviços hospitalares, incluido no total da AIH. |
| 93 | VAL_SP_GES | Valor do complemento do gestor para serviços profissionais, incluido no total da AIH. |
| 94 | VAL_UCI | Valor referente a unidade de cuidados intermediarios (UCI). |
| 95 | MARCA_UCI | Marcador/tipo de unidade de cuidados intermediarios (UCI). |
| 96 | DIAGSEC1 | Diagnostico secundário 1, em código CID. |
| 97 | DIAGSEC2 | Diagnostico secundário 2, em código CID. |
| 98 | DIAGSEC3 | Diagnostico secundário 3, em código CID. |
| 99 | DIAGSEC4 | Diagnostico secundário 4, em código CID. |
| 100 | DIAGSEC5 | Diagnostico secundário 5, em código CID. |
| 101 | DIAGSEC6 | Diagnostico secundário 6, em código CID. |
| 102 | DIAGSEC7 | Diagnostico secundário 7, em código CID. |
| 103 | DIAGSEC8 | Diagnostico secundário 8, em código CID. |
| 104 | DIAGSEC9 | Diagnostico secundário 9, em código CID. |
| 105 | TPDISEC1 | Tipo/classificacao do diagnóstico secundário 1. |
| 106 | TPDISEC2 | Tipo/classificacao do diagnóstico secundário 2. |
| 107 | TPDISEC3 | Tipo/classificacao do diagnóstico secundário 3. |
| 108 | TPDISEC4 | Tipo/classificacao do diagnóstico secundário 4. |
| 109 | TPDISEC5 | Tipo/classificacao do diagnóstico secundário 5. |
| 110 | TPDISEC6 | Tipo/classificacao do diagnóstico secundário 6. |
| 111 | TPDISEC7 | Tipo/classificacao do diagnóstico secundário 7. |
| 112 | TPDISEC8 | Tipo/classificacao do diagnóstico secundário 8. |
| 113 | TPDISEC9 | Tipo/classificacao do diagnóstico secundário 9. |
| 114 | FONTE_ORC | Fonte orçamentária associada ao registro. |


#### Item 3 - Remoção de colunas não utilizadas

Nesta etapa vamos manter apenas as colunas necessárias para o recorte analítico definido:

* competência mensal;
* informações pessoais não identificadoras;
* período internado e permanência em dias;
* diagnósticos e classificações relacionadas a CID.

As demais colunas administrativas, financeiras, identificadoras diretas e de controle da AIH serão removidas.

**Neste passo vamos:**

- **3.1 - Definir as colunas mantidas** conforme o recorte analítico.
- **3.2 - Registrar as colunas removidas** para manter rastreabilidade da redução.
- **3.3 - Gerar o arquivo `2025_reduzido.parquet`** com a base menor para as próximas etapas.


In [5]:
colunas_mantidas = [
    # Competencia
    "MES_CMPT",
    "IDENT",

    # Informações pessoais não identificadoras
    "MUNIC_RES",
    "NASC",
    "SEXO",
    "COD_IDADE",
    "IDADE",
    "NUM_FILHOS",
    "INSTRU",
    "GESTRISCO",
    "CONTRACEP1",
    "CONTRACEP2",
    "RACA_COR",
    "ETNIA",

    # Período internado e permanência em dias
    "DT_INTER",
    "DT_SAIDA",
    "DIAS_PERM",
    "QT_DIARIAS",
    "UTI_MES_IN",
    "UTI_MES_AN",
    "UTI_MES_AL",
    "UTI_MES_TO",
    "UTI_INT_IN",
    "UTI_INT_AN",
    "UTI_INT_AL",
    "UTI_INT_TO",
    "MORTE",

    # Informações sobre tipo de doença / diagnósticos CID
    "PROC_REA",
    "DIAG_PRINC",
    "DIAG_SECUN",
    "CID_NOTIF",
    "CID_ASSO",
    "CID_MORTE",
    "DIAGSEC1",
    "DIAGSEC2",
    "DIAGSEC3",
    "DIAGSEC4",
    "DIAGSEC5",
    "DIAGSEC6",
    "DIAGSEC7",
    "DIAGSEC8",
    "DIAGSEC9",
    "TPDISEC1",
    "TPDISEC2",
    "TPDISEC3",
    "TPDISEC4",
    "TPDISEC5",
    "TPDISEC6",
    "TPDISEC7",
    "TPDISEC8",
    "TPDISEC9",
    "COMPLEX",

    #VALORES
    "VAL_TOT",
    "DIAR_ACOM"
]

colunas_mantidas = [coluna for coluna in colunas_mantidas if coluna in dados.columns]
colunas_remover = [coluna for coluna in dados.columns if coluna not in colunas_mantidas]

df_reduzido = dados[colunas_mantidas].copy()
df_reduzido.to_parquet('2025_reduzido.parquet', index=False)

print(f'Colunas mantidas: {colunas_mantidas}')
print(f'Colunas removidas: {colunas_remover}')
print(f'Arquivo gerado: 2025_reduzido.parquet')
print(f'Shape original: {dados.shape}')
print(f'Shape reduzido: {df_reduzido.shape}')


Colunas mantidas: ['MES_CMPT', 'IDENT', 'MUNIC_RES', 'NASC', 'SEXO', 'COD_IDADE', 'IDADE', 'NUM_FILHOS', 'INSTRU', 'GESTRISCO', 'CONTRACEP1', 'CONTRACEP2', 'RACA_COR', 'ETNIA', 'DT_INTER', 'DT_SAIDA', 'DIAS_PERM', 'QT_DIARIAS', 'UTI_MES_IN', 'UTI_MES_AN', 'UTI_MES_AL', 'UTI_MES_TO', 'UTI_INT_IN', 'UTI_INT_AN', 'UTI_INT_AL', 'UTI_INT_TO', 'MORTE', 'PROC_REA', 'DIAG_PRINC', 'DIAG_SECUN', 'CID_NOTIF', 'CID_ASSO', 'CID_MORTE', 'DIAGSEC1', 'DIAGSEC2', 'DIAGSEC3', 'DIAGSEC4', 'DIAGSEC5', 'DIAGSEC6', 'DIAGSEC7', 'DIAGSEC8', 'DIAGSEC9', 'TPDISEC1', 'TPDISEC2', 'TPDISEC3', 'TPDISEC4', 'TPDISEC5', 'TPDISEC6', 'TPDISEC7', 'TPDISEC8', 'TPDISEC9', 'COMPLEX', 'VAL_TOT', 'DIAR_ACOM']
Colunas removidas: ['UF_ZI', 'ANO_CMPT', 'ESPEC', 'CGC_HOSP', 'N_AIH', 'CEP', 'MARCA_UTI', 'PROC_SOLIC', 'VAL_SH', 'VAL_SP', 'VAL_SADT', 'VAL_RN', 'VAL_ACOMP', 'VAL_ORTP', 'VAL_SANGUE', 'VAL_SADTSR', 'VAL_TRANSP', 'VAL_OBSANG', 'VAL_PED1AC', 'VAL_UTI', 'US_TOT', 'COBRANCA', 'NATUREZA', 'NAT_JUR', 'GESTAO', 'RUBRICA', 'IN

#### Item 4 - Validação da base reduzida

**Neste passo vamos:**

- **4.1 - Conferir as colunas finais** gravadas no arquivo `2025_reduzido.parquet`.
- **4.2 - Conferir o shape final** para validar se a redução preservou a quantidade de registros esperada.


In [6]:
df_reduzido = pd.read_parquet('2025_reduzido.parquet')
df_reduzido.columns

Index(['MES_CMPT', 'IDENT', 'MUNIC_RES', 'NASC', 'SEXO', 'COD_IDADE', 'IDADE',
       'NUM_FILHOS', 'INSTRU', 'GESTRISCO', 'CONTRACEP1', 'CONTRACEP2',
       'RACA_COR', 'ETNIA', 'DT_INTER', 'DT_SAIDA', 'DIAS_PERM', 'QT_DIARIAS',
       'UTI_MES_IN', 'UTI_MES_AN', 'UTI_MES_AL', 'UTI_MES_TO', 'UTI_INT_IN',
       'UTI_INT_AN', 'UTI_INT_AL', 'UTI_INT_TO', 'MORTE', 'PROC_REA',
       'DIAG_PRINC', 'DIAG_SECUN', 'CID_NOTIF', 'CID_ASSO', 'CID_MORTE',
       'DIAGSEC1', 'DIAGSEC2', 'DIAGSEC3', 'DIAGSEC4', 'DIAGSEC5', 'DIAGSEC6',
       'DIAGSEC7', 'DIAGSEC8', 'DIAGSEC9', 'TPDISEC1', 'TPDISEC2', 'TPDISEC3',
       'TPDISEC4', 'TPDISEC5', 'TPDISEC6', 'TPDISEC7', 'TPDISEC8', 'TPDISEC9',
       'COMPLEX', 'VAL_TOT', 'DIAR_ACOM'],
      dtype='str')

In [7]:
df_reduzido.shape

(925240, 54)